In [1]:
import pandas as pd
from pathlib import Path
import re
from typing import List, Tuple, cast

# Configuración de rutas
proyecto_root = Path("/home/agusriscos/proyectos/ARPTools")
carpeta_grafos = proyecto_root / "data" / "16dic25" / "grafos"

print(f"Buscando archivos de métricas en: {carpeta_grafos}")


Buscando archivos de métricas en: /home/agusriscos/proyectos/ARPTools/data/16dic25/grafos


In [2]:
def extraer_imf_y_transformacion(ruta_archivo: Path) -> Tuple[str, str]:
    """
    Extrae el IMF y la transformación de la ruta del archivo de métricas.
    
    Parameters
    ----------
    ruta_archivo : Path
        Ruta completa al archivo de métricas.
        
    Returns
    -------
    Tuple[str, str]
        Tupla con (imf, transformacion) extraídos de la ruta.
        
    Examples
    --------
    >>> ruta = Path("data/16dic25/grafos/nvg/imf_1/graph_metrics/metricas_grafo.csv")
    >>> imf, transformacion = extraer_imf_y_transformacion(ruta)
    >>> print(imf, transformacion)
    imf_1 nvg
    """
    # La estructura es: grafos/{transformacion}/{imf}/graph_metrics/metricas_grafo.csv
    partes = ruta_archivo.parts
    # Buscar el índice de 'grafos' en la ruta
    try:
        idx_grafos = partes.index('grafos')
        transformacion = partes[idx_grafos + 1]
        imf = partes[idx_grafos + 2]
        return imf, transformacion
    except (ValueError, IndexError):
        # Si no encontramos la estructura esperada, intentar extraer del nombre del archivo
        nombre_archivo = ruta_archivo.name
        # Patrón: grafo_{transformacion}_{imf}_metadata.csv
        match = re.match(r'grafo_(\w+)_(.+?)_metadata\.csv', nombre_archivo)
        if match:
            return match.group(2), match.group(1)
        raise ValueError(f"No se pudo extraer IMF y transformación de: {ruta_archivo}")


def leer_todas_metricas_grafos(carpeta_base: Path) -> pd.DataFrame:
    """
    Lee todos los archivos de métricas de grafos y los combina en un único DataFrame.
    
    Agrega dos columnas nuevas: 'imf' y 'transformacion' indicando a qué
    IMF y transformación pertenece cada registro de métricas.
    
    Parameters
    ----------
    carpeta_base : Path
        Ruta base donde se encuentran los archivos de métricas.
        Estructura esperada: {carpeta_base}/{transformacion}/{imf}/graph_metrics/metricas_grafo.csv
        
    Returns
    -------
    pd.DataFrame
        DataFrame con todas las métricas combinadas, incluyendo las columnas
        'imf' y 'transformacion'.
        
    Examples
    --------
    >>> carpeta = Path("data/16dic25/grafos")
    >>> df_completo = leer_todas_metricas_grafos(carpeta)
    >>> print(df_completo.head())
    """
    archivos_metricas = list(carpeta_base.rglob("graph_metrics/metricas_grafo.csv"))
    
    if not archivos_metricas:
        raise FileNotFoundError(
            f"No se encontraron archivos de métricas en: {carpeta_base}"
        )
    
    print(f"Encontrados {len(archivos_metricas)} archivos de métricas")
    
    lista_dataframes = []
    
    for archivo in archivos_metricas:
        try:
            # Leer el archivo CSV
            df_temp = pd.read_csv(archivo)
            
            # Extraer IMF y transformación de la ruta
            imf, transformacion = extraer_imf_y_transformacion(archivo)
            
            # Agregar columnas nuevas
            df_temp['imf'] = imf
            df_temp['transformacion'] = transformacion
            
            lista_dataframes.append(df_temp)
            
        except Exception as e:
            print(f"Error al procesar {archivo}: {e}")
            continue
    
    if not lista_dataframes:
        raise ValueError("No se pudo leer ningún archivo de métricas")
    
    # Combinar todos los DataFrames
    df_completo = cast(pd.DataFrame, pd.concat(lista_dataframes, ignore_index=True))
    
    # Reordenar columnas para que 'imf' y 'transformacion' estén al principio
    columnas = ['imf', 'transformacion'] + [col for col in df_completo.columns 
                                            if col not in ['imf', 'transformacion']]
    df_completo = df_completo[columnas]
    
    return df_completo  # type: ignore[return-value]


In [3]:
# Leer todos los archivos de métricas
df_metricas_completo = leer_todas_metricas_grafos(carpeta_grafos)

print(f"\nDataFrame completo creado:")
print(f"  - Total de registros: {len(df_metricas_completo)}")
print(f"  - Total de columnas: {len(df_metricas_completo.columns)}")
print(f"\nColumnas: {list(df_metricas_completo.columns)}")
print(f"\nPrimeras filas:")
df_metricas_completo.head()


Encontrados 32 archivos de métricas

DataFrame completo creado:
  - Total de registros: 32
  - Total de columnas: 35

Columnas: ['imf', 'transformacion', 'num_nodos', 'num_enlaces', 'densidad', 'num_componentes', 'diametro', 'radio', 'excentricidad_promedio', 'grado_min', 'grado_max', 'grado_promedio', 'grado_mediana', 'grado_std', 'clustering_promedio', 'clustering_min', 'clustering_max', 'clustering_mediana', 'clustering_std', 'degree_promedio', 'degree_mediana', 'degree_max', 'degree_min', 'betweenness_promedio', 'betweenness_mediana', 'betweenness_max', 'betweenness_min', 'closeness_promedio', 'closeness_mediana', 'closeness_max', 'closeness_min', 'eigenvector_promedio', 'eigenvector_mediana', 'eigenvector_max', 'eigenvector_min']

Primeras filas:


,imf,transformacion,num_nodos,num_enlaces,densidad,num_componentes,diametro,radio,excentricidad_promedio,grado_min,...,betweenness_max,betweenness_min,closeness_promedio,closeness_mediana,closeness_max,closeness_min,eigenvector_promedio,eigenvector_mediana,eigenvector_max,eigenvector_min
0,imf_7,hvg,3490,6757,0.001110,1,347,174,256.016332,1,...,0.580473,0.0,0.008377,0.008170,0.012259,0.004756,0.001767,9.413112e-12,0.555506,2.616952e-30
1,imf_8,hvg,3490,6170,0.001013,1,1017,509,721.501146,1,...,0.640908,0.0,0.003403,0.003438,0.004892,0.001637,0.001755,7.090989e-10,0.560653,5.157293e-24
2,imf_2,hvg,3490,6962,0.001144,1,26,13,20.145845,2,...,0.550116,0.0,0.094670,0.093437,0.167362,0.051215,0.005791,1.333057e-03,0.324320,1.285955e-09
3,imf_9,hvg,3490,5293,0.000869,1,1732,866,1230.811175,1,...,0.623874,0.0,0.001983,0.001997,0.002741,0.001083,0.001730,1.128866e-10,0.466577,2.298112e-27
4,residuo,hvg,3490,3489,0.000573,1,3489,1745,2617.000000,1,...,0.500143,0.0,0.000900,0.000917,0.001146,0.000573,0.016920,1.694909e-02,0.016949,3.017715e-03


In [4]:
# Resumen por transformación e IMF
print("Resumen por transformación:")
print(df_metricas_completo.groupby('transformacion').size())
print("\nResumen por IMF:")
print(df_metricas_completo.groupby('imf').size())
print("\nResumen combinado (transformación x IMF):")
print(df_metricas_completo.groupby(['transformacion', 'imf']).size().unstack(fill_value=0))


Resumen por transformación:
transformacion
hvg            11
nvg            10
recurrencia    11
dtype: int64

Resumen por IMF:
imf
imf_1      3
imf_10     3
imf_2      3
imf_3      3
imf_4      3
imf_5      3
imf_6      3
imf_7      3
imf_8      3
imf_9      3
residuo    2
dtype: int64

Resumen combinado (transformación x IMF):
imf             imf_1  imf_10  imf_2  imf_3  imf_4  imf_5  imf_6  imf_7  \
transformacion                                                            
hvg                 1       1      1      1      1      1      1      1   
nvg                 1       1      1      1      1      1      1      1   
recurrencia         1       1      1      1      1      1      1      1   

imf             imf_8  imf_9  residuo  
transformacion                         
hvg                 1      1        1  
nvg                 1      1        0  
recurrencia         1      1        1  


In [5]:
# Guardar el DataFrame completo
archivo_salida = proyecto_root / "data" / "16dic25" / "analisis_grafo_completo.csv"
df_metricas_completo.to_csv(archivo_salida, index=False)
print(f"✓ Tabla completa guardada en: {archivo_salida}")


✓ Tabla completa guardada en: /home/agusriscos/proyectos/ARPTools/data/16dic25/analisis_grafo_completo.csv
